In [48]:
library(arrow)

In [44]:
lineages <- c('Adipocyte', 'B_plasma', 'Endothelial', 'Low quality', 'Mast', 'Mural',
              'Myeloid', 'Myeloid+B_plasma', 'Myeloid+Stromal', 'Neuronal_glial', 'Osteoclast',
              'Proliferating', 'Stromal', 'T_NK')

version <- 'EDP1-EDP2-ARB'
src_dir <- paste0('out_rds/', version)
dst_dir <- paste0('/data/srlab/AMP_collab/data/early_disease_synovium/xenium/combined/', version)

dir.create(dst_dir,                        showWarnings = FALSE, recursive = TRUE)
dir.create(paste0(dst_dir, '/lineages'),   showWarnings = FALSE, recursive = TRUE)

In [2]:
# =============================================================================
# Copy lineage RDS files
# =============================================================================

message("Copying ", length(lineages), " lineages from ", src_dir, "/lineages to ", dst_dir, "/lineages")

for (i in seq_along(lineages)) {
    lineage <- lineages[[i]]
    fname   <- paste0(lineage, '.rds')
    src     <- file.path(src_dir, 'lineages', fname)
    dst     <- file.path(dst_dir, 'lineages', fname)
    file.copy(src, dst, overwrite = TRUE)
    message(sprintf("  [%d/%d] copied: %s", i, length(lineages), fname))
}

message("Done.")

Copying 14 lineages from out_rds/EDP1-EDP2-ARB/lineages to /data/srlab/AMP_collab/data/early_disease_synovium/xenium/combined/EDP1-EDP2-ARB/lineages

  [1/14] copied: Adipocyte.rds

  [2/14] copied: B_plasma.rds

  [3/14] copied: Endothelial.rds

  [4/14] copied: Low quality.rds

  [5/14] copied: Mast.rds

  [6/14] copied: Mural.rds

  [7/14] copied: Myeloid.rds

  [8/14] copied: Myeloid+B_plasma.rds

  [9/14] copied: Myeloid+Stromal.rds

  [10/14] copied: Neuronal_glial.rds

  [11/14] copied: Osteoclast.rds

  [12/14] copied: Proliferating.rds

  [13/14] copied: Stromal.rds

  [14/14] copied: T_NK.rds

Done.



In [45]:
# add tile id and tile cluster id into cell metadata
cellmeta <- readRDS(paste0(src_dir, '/allcells_qc_harmumapclust_lineage[metadata].rds'))
cellstotiles <- readRDS(paste0(src_dir, '/allcells_qc_harmumapclust_lineage[cellstotiles].rds'))
tiles <- readRDS(paste0(src_dir, '/allcells_qc_harmumapclust_lineage[tiles].rds'))

cellstotiles$tile_cluster <- tiles@meta.data[cellstotiles$tile_id, "seurat_clusters"]
write_parquet(cellstotiles %>% tibble::rownames_to_column("cell_id"), paste0(dst_dir, '/allcells_qc_harmumapclust_lineage[metadata-tiles].parquet'))

In [47]:
tiles
# # write the tile metadata
# file.copy(
#     paste0(src_dir, '/allcells_qc_harmumapclust_lineage[tiles].rds'),
#     paste0(dst_dir, '/allcells_qc_harmumapclust_lineage[tiles].rds')
# )

An object of class Seurat 
5100 features across 181471 samples within 1 assay 
Active assay: RNA (5100 features, 0 variable features)
 1 layer present: counts
 2 dimensional reductions calculated: pca, humap

In [35]:
# convert harmony and umap coordinates to parquet and write
library(dplyr)

files <- c(paste0(src_dir, '/allcells_qc_harmumapclust_lineage[humap].rds'),
           paste0(src_dir, '/allcells_qc_harmumapclust_lineage[harmony].rds'))

message("Converting ", length(files), " files from ", src_dir, " to ", dst_dir)

for (i in seq_along(files)) {
    file  <- files[[i]]
    fname <- basename(file)
    fname <- sub("\\.rds$", ".parquet", fname)
    dst   <- file.path(dst_dir, fname)
    data  <- readRDS(file)
    write_parquet(as.data.frame(data) %>% tibble::rownames_to_column("cell_id"), dst)
    message(sprintf("  [%d/%d] converted: %s", i, length(files), fname))
}

message("Done.")

Converting 2 files from out_rds/EDP1-EDP2-ARB to /data/srlab/AMP_collab/data/early_disease_synovium/xenium/combined/EDP1-EDP2-ARB

  [1/2] converted: allcells_qc_harmumapclust_lineage[humap].parquet

  [2/2] converted: allcells_qc_harmumapclust_lineage[harmony].parquet

Done.



In [ ]:
# copy tile shapes file


In [42]:
# remove metadata
xen <- readRDS(paste0(src_dir, '/allcells_qc_harmumapclust_lineage.rds'))
xen@meta.data <- data.frame(row.names = colnames(xen))
saveRDS(xen, paste0(dst_dir, '/allcells_qc_harmumapclust_lineage[seurat_nometadata].rds'))